# Econ 5200 PS2 — The Underpowered Portfolio

**ECON 5200 · Causal Machine Learning & Applied Analytics** · Topics 5-8 · 100 points · your role: Experimentation lead, consulting engagement

This is the **starter**: the structure of the set in rubric order. Fill the empty cells; keep every heading so the grader finds each step. The brief on Canvas has the full text of every step, the numbers you must hit and the rubric. Every claim goes in a text cell: the reader is experimentation lead's audience, not a grader with a kernel.

**AI is not allowed in Phases 1–3.** Generative AI is **prohibited** in Phases 1 through 3. AI is authorised and required in Phase 4. You must be able to defend every line under questioning.

## Setup

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)     # one generator, so every draw reruns the same way
pd.set_option('display.precision', 3)

import os
os.makedirs('src', exist_ok=True)   # the .py module the brief asks for goes here

## Phase 1: Reconstruct the Portfolio (30 pts)

### Step 1.1: Simulate the experiment programme (15 pts)

In [4]:
rng = np.random.default_rng(3)
n_experiments = 200

share = 0.30
outcome_sd = 1.0

real = rng.random(n_experiments) < share

truth = np.where(
    real,
    rng.normal(0.30, 0.10, size=n_experiments),
    0.0
)

sample_size = rng.integers(40, 400, size=n_experiments)
standard_error = outcome_sd * np.sqrt(2 / sample_size)

effect_estimate = truth + rng.normal(
    0, standard_error, size=n_experiments
)

experiments = pd.DataFrame({
    "Truth": truth,
    "Sample_Size": sample_size,
    "Standard_Error": standard_error,
    "Effect_Estimate": effect_estimate
})

print("Number of experiments:", len(experiments))
print("Mean true effect:", experiments["Truth"].mean())
print("Mean estimated effect:", experiments["Effect_Estimate"].mean())

Number of experiments: 200
Mean true effect: 0.0865269296771451
Mean estimated effect: 0.1023687777766047


*Your answer here.*
I assumed that 30% of all experiments will have  real effect drawn from a normal distribution and the other 70% will have no true effect. Each experiment's true effect if listed out before generating an observed estimate. Then I added random sampling noise determined by the standard error and sampling size to each true effect.

### Step 1.2: Apply the client's rule (15 pts)

In [5]:
experiments["z_score"] = (
    experiments["Effect_Estimate"] / experiments["Standard_Error"])

experiments["p_value"] = 2 * stats.norm.sf(
    np.abs(experiments["z_score"]))

experiments["Ship"] = experiments["p_value"] < 0.05

shipped = experiments[experiments["Ship"]]

print("Experiments shipped:", len(shipped))
print("Mean estimated effect:", round(shipped["Effect_Estimate"].mean(), 3))
print("Mean true effect:", round(shipped["Truth"].mean(), 3))
print(
    "False-ship rate:",
    round((shipped["Truth"] == 0).mean() * 100, 1), "%")

Experiments shipped: 53
Mean estimated effect: 0.321
Mean true effect: 0.289
False-ship rate: 13.2 %


*Your answer here.*

Next we calculate the z score for each experiment by dividing the effect estimate by the standard error. You use this number to calculate the p value which tells us if the true effect is 0. If a p value is found to be under 0.05 then the experiment is shiped. Next the number of experiments that were shipped, the mean true effect and mean estimated effect across the 200 experiments is calculated.

## Phase 2: The Winner's Curse (35 pts)

### Step 2.1: Quantify the exaggeration (20 pts)

In [6]:
shipped = experiments[experiments["p_value"] < 0.05].copy()

mean_estimated_effect = shipped["Effect_Estimate"].mean()
mean_true_effect = shipped["Truth"].mean()

absolute_inflation = mean_estimated_effect - mean_true_effect

relative_inflation = absolute_inflation / mean_true_effect

print("Shipped:", len(shipped))
print("Mean estimated effect:", round(mean_estimated_effect, 4))
print("Mean true effect:", round(mean_true_effect, 4))
print("Inflation:", round(absolute_inflation, 4))

z_critical = stats.norm.ppf(0.975)
effect_to_se = experiments["Truth"] / experiments["Standard_Error"]

experiments["Power"] = (stats.norm.cdf(-z_critical - effect_to_se)
 + stats.norm.sf(z_critical - effect_to_se))

experiments["Power_Tier"] = pd.cut(
    experiments["Power"],
    bins=[-np.inf, 0.40, 0.80, np.inf],
    labels=["Low", "Medium", "High"])

shipped = experiments[experiments["p_value"] < 0.05].copy()

tier_results = shipped.groupby(
    "Power_Tier", observed=True).agg(
    Number_Shipped=("Truth", "size"),
    Mean_True_Effect=("Truth", "mean"),
    Mean_Estimated_Effect=("Effect_Estimate", "mean"))

tier_results["Absolute_Inflation"] = (
    tier_results["Mean_Estimated_Effect"]
    - tier_results["Mean_True_Effect"])


print("\nInflation by power tier:")
print(tier_results.round(4))

Shipped: 53
Mean estimated effect: 0.3209
Mean true effect: 0.2891
Inflation: 0.0318

Inflation by power tier:
            Number_Shipped  Mean_True_Effect  Mean_Estimated_Effect  \
Power_Tier                                                            
Low                      9             0.036                 -0.007   
Medium                  10             0.307                  0.393   
High                    34             0.351                  0.387   

            Absolute_Inflation  
Power_Tier                      
Low                     -0.043  
Medium                   0.086  
High                     0.036  


*Your answer here.*

this compares the average estimated effect of the shipped experiments against their average true effect to measure inflation. Next I calculated the percentage of shiped experiments whose true effect is 0 or "false ship rate"

### Step 2.2: The bootstrap where the formula fails (15 pts)

In [7]:
realized_effects = shipped["Truth"].to_numpy()


rn = np.random.default_rng(42)
n_bootstrap = 10000
bootstrap_mean = []

for _ in range(n_bootstrap):
  sample = rn.choice(
      realized_effects,
      size=len(realized_effects),
      replace=True)
  bootstrap_mean.append(sample.mean())

bootstrap_mean = np.array(bootstrap_mean)

print("Realized return:", round(realized_effects.mean(), 4))
print(
    "Manual bootstrap 95% interval:",
    np.round(np.percentile(bootstrap_mean, [2.5, 97.5]), 4))

result = stats.bootstrap(
    (realized_effects,),
    np.mean,
    confidence_level=0.95,
    n_resamples=10000,
    method="BCa",
    rng=np.random.default_rng(42))

print("SciPy BCa 95% interval:",
      np.round([result.confidence_interval.low,
                result.confidence_interval.high], 4))

Realized return: 0.2891
Manual bootstrap 95% interval: [0.2498 0.3254]
SciPy BCa 95% interval: [0.2489 0.3249]


*Your answer here.*

The manual bootstrap interval and the SciPy BCa interval are very close to eachother. this could be because my manual interval does not account for any bias within the bootstrap distribution that the SciPy calculation may have.

## Phase 3: A Rule They Can Adopt (25 pts)

### Step 3.1: Design the alternative (20 pts)

In [13]:
import numpy as np
import pandas as pd
from scipy import stats

# Client's rule: ship if p < 0.05
client_shipped = experiments[(experiments["p_value"] < 0.05) & (experiments["Effect_Estimate"] > 0)]

# Your proposed rule: ship if p < 0.01 and positive effect
proposed_shipped = experiments[(experiments["p_value"] < 0.01) & (experiments["Effect_Estimate"] > 0)]

print("Client rule")
print("Experiments shipped:", len(client_shipped))
print("False-ship rate:", round((client_shipped["Truth"] == 0).mean() * 100, 2), "%")

print("\nMy rule")
print("Experiments shipped:", len(proposed_shipped))
print("False-ship rate:", round((proposed_shipped["Truth"] == 0).mean() * 100, 2), "%")

Client rule
Experiments shipped: 48
False-ship rate: 4.17 %

My rule
Experiments shipped: 42
False-ship rate: 0.0 %


*Your answer here.*

For my rule, I decided to only ship if the pvalue was less than 0.01 and had a positive effect. the number of shipped experiments went from 48 to 42 and the false ship rate was 0. This could be helpful however it rejected some experiments that would have been helpful. This rule was likely not as good as the clients rule.

### Step 3.2: Ship the module (5 pts, its own rubric row)

The module the brief asks for. Write it with `%%writefile`, then import it back in the next cell and prove it runs.

In [26]:
%%writefile src/experiment_portfolio.py
"""Experiment-portfolio simulator and decision rules (ECON 5200 PS2)."""
from __future__ import annotations
import numpy as np
import pandas as pd
from scipy import stats
def simulate_portfolio(n: int, seed: int, **kwargs):
    """Return the experiment log with the true effect attached."""
    rn = np.random.default_rng(seed)

    share = kwargs.get("share", 0.30)
    effect_mean = kwargs.get("effect_mean", 0.30)
    effect_sd = kwargs.get("effect_sd", 0.10)
    sample_size_low = kwargs.get("sample_size_low", 40)
    sample_size_high = kwargs.get("sample_size_high", 400)
    outcome_sd = kwargs.get("outcome_sd", 1.0)

    real = rn.random(n) < share

    truth = np.where(
        real,
        rn.normal(effect_mean, effect_sd, size=n), 0.0)

    sample_size = rn.integers(
        sample_size_low, sample_size_high + 1, size=n)
    standard_error = outcome_sd * np.sqrt(2 / sample_size)

    estimate = truth + rn.normal(0, standard_error)

    log = pd.DataFrame({
        "Truth": truth,
        "Sample_Size": sample_size,
        "Outcome_SD": outcome_sd,
        "Standard_Error": standard_error,
        "Effect_Estimate": estimate})

    z_score = log["Effect_Estimate"] / log["Standard_Error"]
    log["p_value"] = 2 * stats.norm.sf(np.abs(z_score))

    alpha = 0.05
    minimum_effect = 0.30
    z_critical = stats.norm.ppf(1 - alpha / 2)

    noncentrality = minimum_effect / log["Standard_Error"]

    log["Power"] = (
        stats.norm.cdf(-z_critical - noncentrality)
        + stats.norm.sf(z_critical - noncentrality))
    return log

def apply_rule(log, rule):
    """Ship decisions under `rule`: the p < 0.05 threshold or p < 0.01 threshold."""
    if rule == "client":
        return (log["p_value"] < 0.05) & (log["Effect_Estimate"] > 0)

    if rule == "proposed":
        return (log["p_value"] < 0.01) & (log["Effect_Estimate"] > 0)

    raise ValueError(
        "Unknown rule. Use 'client' or 'proposed'.")

def evaluate(log, rule):
    """Realised true return, ship count and false-ship rate under `rule`."""
    shipped = log[apply_rule(log, rule)]
    ship_count = len(shipped)

    if ship_count == 0:
        return {
            "Realized Return": 0.0,
            "Number Shipped": 0,
            "False-Ship Rate": np.nan,
            "Average True Effect": np.nan,
            "Average Estimated Effect": np.nan}

    return {
        "Realized Return": shipped["Truth"].sum(),
        "Number Shipped": ship_count,
        "False-Ship Rate": (shipped["Truth"] == 0).mean(),
        "Average True Effect": shipped["Truth"].mean(),
        "Average Estimated Effect": shipped["Effect_Estimate"].mean()}

if __name__ == "__main__":
    log = simulate_portfolio(200, seed=3)

    print("Client rule:", evaluate(log, "client"))
    print("Proposed rule:", evaluate(log, "proposed"))

    same_log = simulate_portfolio(200, seed=3)
    assert log.equals(same_log), "Same seed should reproduce the results."

    client_ships = apply_rule(log, "client")
    assert (client_ships == ((log["p_value"] < 0.05) & (log["Effect_Estimate"] > 0))).all()

    proposed_ships = apply_rule(log, "proposed")
    assert (proposed_ships == ((log["p_value"] < 0.01) & (log["Effect_Estimate"] > 0))).all()

    assert 0 <= client_ships.sum() <= 200

    print("All assertions passed.")

Overwriting src/experiment_portfolio.py


In [27]:
import os, sys
import importlib
from src import experiment_portfolio as ep

importlib.reload(ep)

log = ep.simulate_portfolio(200, seed=3)

print("Client rule:", ep.evaluate(log, "client"))
print("Proposed rule:", ep.evaluate(log, "proposed"))

small_log = ep.simulate_portfolio(10, seed=1)
ships = ep.apply_rule(small_log, "client")
results = ep.evaluate(small_log, "client")

assert len(small_log) == 10
assert len(ships) == 10
assert isinstance(results, dict)

print("All three functions run successfully")

Client rule: {'Realized Return': np.float64(15.322161950114305), 'Number Shipped': 48, 'False-Ship Rate': np.float64(0.041666666666666664), 'Average True Effect': np.float64(0.319211707294048), 'Average Estimated Effect': np.float64(0.3846277981533251)}
Proposed rule: {'Realized Return': np.float64(14.000122724915032), 'Number Shipped': 42, 'False-Ship Rate': np.float64(0.0), 'Average True Effect': np.float64(0.3333362553551198), 'Average Estimated Effect': np.float64(0.39251634177301603)}
All three functions run successfully with updated parameters.


In [36]:
import os
# Check if the folder and file exists
if os.path.exists('src/experiment_portfolio.py'):
    print("✅ Success! 'src/experiment_portfolio.py' exists in your workspace.")
    print("Files inside 'src/':", os.listdir('src'))
else:
    print("❌ 'src' folder or python file not found. Let's create it now...")
    os.makedirs('src', exist_ok=True)

# Force a file tree refresh inside Google Colab's interface
from google.colab import files

✅ Success! 'src/experiment_portfolio.py' exists in your workspace.
Files inside 'src/': ['experiment_portfolio.py', '__pycache__']


## Phase 4: AI Expansion — The Co-Pilot Era (10 pts: 7 for the AI row, 3 for the CFO memo)

**Status: AI usage AUTHORISED and REQUIRED.** Use the **P.R.I.M.E. framework**.

### Task 4.1: The sequential design

In [28]:
"""Always-valid sequential test: two-sided mixture SPRT (Gaussian mixing prior).
numpy + scipy only.

Stream model: each observation is a paired difference d_i = treatment_i - control_i,
d_i ~ N(theta, sigma_d^2) with sigma_d known. H0: theta = 0.
Mixing prior on theta: N(0, tau^2).

  Lambda_n = sqrt(s2/(s2 + n tau^2)) * exp( n^2 tau^2 dbar^2 / (2 s2 (s2 + n tau^2)) )

Under H0, Lambda_n is a nonnegative martingale with mean 1, so by Ville's inequality
P(sup_n Lambda_n >= 1/alpha) <= alpha.  Reject at the first n with Lambda_n >= 1/alpha.
"""
import numpy as np


def msprt(stream, alpha=0.05, sigma_d=1.0, tau=None):
    """Return the per-look log (columns as arrays, one entry per look).

    stream  : 1-D array of observations (paired differences), or 2-D (n, 2) array
              of (control, treatment) which is converted to differences.
    sigma_d : known SD of one observation (the difference).
    tau     : SD of the mixing prior on the effect; default 0.5 * sigma_d.
    """
    x = np.asarray(stream, dtype=float)
    if x.ndim == 2:
        x = x[:, 1] - x[:, 0]
    n = np.arange(1, x.size + 1, dtype=float)
    s2 = sigma_d ** 2
    tau = 0.5 * sigma_d if tau is None else tau
    t2 = tau ** 2
    mean = np.cumsum(x) / n
    log_lam = (0.5 * np.log(s2 / (s2 + n * t2))
               + (n ** 2 * t2 * mean ** 2) / (2 * s2 * (s2 + n * t2)))
    run_max = np.maximum.accumulate(log_lam)
    p_av = np.minimum(1.0, np.exp(-run_max))          # always-valid p-value
    reject = run_max >= np.log(1.0 / alpha)           # monotone: stays True once crossed
    return {"look": n.astype(int), "mean_diff": mean, "log_lambda": log_lam,
            "p_always_valid": p_av, "stop": reject}


def first_stop(log):
    idx = np.flatnonzero(log["stop"])
    return int(idx[0]) + 1 if idx.size else None


def naive_fixed_z(stream, alpha=0.05, sigma_d=1.0):
    """The wrong way: fixed-horizon z-test applied at every look."""
    from scipy.stats import norm
    x = np.asarray(stream, float)
    n = np.arange(1, x.size + 1)
    z = np.cumsum(x) / (sigma_d * np.sqrt(n))
    return np.abs(z) > norm.ppf(1 - alpha / 2)


if __name__ == "__main__":
    SEED, N_STREAMS, N_LOOKS, ALPHA, SIGMA = 20260427, 1000, 5000, 0.05, 1.0
    rng = np.random.default_rng(SEED)

    # --- pure null ---
    fp_m = fp_n = 0
    for _ in range(N_STREAMS):
        s = rng.normal(0.0, SIGMA, N_LOOKS)
        fp_m += msprt(s, ALPHA, SIGMA)["stop"][-1]
        fp_n += naive_fixed_z(s, ALPHA, SIGMA).any()
    print(f"[null] nominal alpha            : {ALPHA:.3f}")
    print(f"[null] mSPRT false positive rate: {fp_m/N_STREAMS:.3f}  ({fp_m}/{N_STREAMS})")
    print(f"[null] naive z-every-look FPR   : {fp_n/N_STREAMS:.3f}  ({fp_n}/{N_STREAMS})")
    se = np.sqrt(ALPHA * (1 - ALPHA) / N_STREAMS)
    print(f"[null] MC std. error at 5%      : {se:.4f}")

    # --- portfolio of 200 tests with known effects ---
    rng = np.random.default_rng(SEED + 1)
    eff = np.where(rng.random(200) < 0.5, 0.0, rng.choice([0.03, 0.05, 0.08, 0.12], 200))
    res = []
    for th in eff:
        s = rng.normal(th, SIGMA, N_LOOKS)
        res.append(first_stop(msprt(s, ALPHA, SIGMA)))
    res = np.array([r if r else 0 for r in res]); rej = res > 0
    null = eff == 0
    print(f"[portfolio] {null.sum()} nulls, {(~null).sum()} true effects")
    print(f"[portfolio] false positives: {rej[null].sum()}/{null.sum()}")
    print(f"[portfolio] power          : {rej[~null].mean():.3f}")
    print(f"[portfolio] median stop look among detected effects: {int(np.median(res[rej & ~null]))}")


[null] nominal alpha            : 0.050
[null] mSPRT false positive rate: 0.040  (40/1000)
[null] naive z-every-look FPR   : 0.620  (620/1000)
[null] MC std. error at 5%      : 0.0069
[portfolio] 93 nulls, 107 true effects
[portfolio] false positives: 3/93
[portfolio] power          : 0.710
[portfolio] median stop look among detected effects: 1022


In [33]:
import numpy as np

N = 1000
T = 5000
alpha = 0.05
rng = np.random.default_rng(7)

msprt_rejections = 0
naive_rejections = 0

for _ in range(N):
    # Pure-null data: both groups have the same true mean
    control = rng.normal(0, 1, T)
    treatment = rng.normal(0, 1, T)
    differences = treatment - control

    # Always-valid sequential test (msprt) on paired differences
    # The differences have SD = sqrt(Var(C) + Var(T)) = sqrt(2)
    result = msprt(
        differences,
        alpha=alpha,
        sigma_d=np.sqrt(2),
        tau=0.5 * np.sqrt(2)
    )
    msprt_rejections += result["stop"][-1]

    # Naive test at every look
    naive_rejections += naive_fixed_z(
        differences,
        alpha=alpha,
        sigma_d=np.sqrt(2)
    ).any()

msprt_fpr = msprt_rejections / N
naive_fpr = naive_rejections / N
mc_se = np.sqrt(alpha * (1 - alpha) / N)

print(f"Nominal alpha:              {alpha:.3f}")
print(f"mSPRT false positive rate:  {msprt_fpr:.3f} ({msprt_rejections}/{N})")
print(f"Naive z-test FPR:           {naive_fpr:.3f} ({naive_rejections}/{N})")
print(f"Monte Carlo SE at alpha:    {mc_se:.4f}")

assert 0 <= msprt_fpr <= 1
assert 0 <= naive_fpr <= 1

Nominal alpha:              0.050
mSPRT false positive rate:  0.032 (32/1000)
Naive z-test FPR:           0.641 (641/1000)
Monte Carlo SE at alpha:    0.0069


*Your answer here.*
I used Claude to code for an mSPRT test and compared it to the normal z-test with 1000 simulations. The false positive rate for mSPRT was 0.032 and the z-test was 0.641. These values are very different. THis could be because of repeatedly testing the data could lead to more false positive detected.

### Task 4.2: The CFO memo

*Your answer here.*

Out of 53 experiments shipped under the current rules, 86.8% had a real positive effect and 13.2% (7 experiments) had no real effect. This shows that while significant results are mostly correct, it might not always be a total win.

Across the 200 experiments, the current client rule (p < 0.05) brings in 15.32 units of true revenue. However, the math overstates this return by 1.69 units because the estimates say we made 17.01 units.

If we switch to the proposed rule (p < 0.01), only 42 experiments are shipped. While this cuts our false-ship rate to 0.0%, we miss out on good projects and only make 14.00 units of true revenue.

My recommendation is to keep the original p < 0.05 rule because it makes us more total revenue. The stricter rule leaves money on the table by rejecting good projects.

### AI Transcript

Paste **both prompt versions exactly as you sent them** and the model's replies **exactly as they came back**, then one line on each change you made. Name the model and the date. For 5200 the transcript lives in `ai-appendix.md` in your repository; link it here.

*Model and date:*
Claude 10/26 Sonnet 5.5

*Version 1 prompt:*
[Prep] Act as an experimentation platform engineer who has implemented sequential testing in production.

[Request] I have a simulated portfolio of 200 A/B tests with known true effects. Implement an always-valid sequential test -- a mixture sequential probability ratio test or an alpha-spending boundary -- that allows continuous monitoring without inflating the Type I error rate. Give me a function that takes a stream of observations and returns the stopping decision at each look.

[Iterate] numpy and scipy only. Reuse my log structure. The function must be testable against a pure-null stream.

[Mechanism Check] Explain why the naive approach of testing at every look inflates Type I error, and precisely what the always-valid construction gives up in exchange for continuous monitoring.

[Evaluate] Run it on 1,000 pure-null streams and report the empirical false positive rate. If it is not close to 5%, tell me what is wrong.

*Reply:*
I’ll write it, run it, and check the numbers. One note first: your message mentions “my log structure” and the 200-test portfolio, but nothing was attached. I’ll use a plain per-look log (dict of arrays) that’s easy to remap, and simulate a portfolio.

The implementation is in msprt.py. The msprt_stream(x_ctrl, x_trt, alpha, tau2, sigma2) function takes paired observation streams and returns a per-look log with n, diff, log_lambda, p_always_valid, ci_lo, ci_hi, reject and stop_at. It uses numpy and scipy only. It is a two-sample mSPRT with a N(0, τ²) mixture prior on the effect. It rejects at the first look where Λₙ ≥ 1/α, and p_always_valid is the running minimum of 1/Λ. Since I didn’t have your log schema, rename the dict keys to match it.

Null results (1,000 streams × 5,000 looks, α = 0.05) Method Empirical FPR mSPRT, known σ² 3.0% mSPRT, plug-in σ², starting at n=30 3.5% mSPRT, plug-in σ², starting at n=2 16.2% (broken) Naive z-test at every look 70.2%

The 3.0% and 3.5% are below 5% by design, not by error. Ville’s inequality guarantees P(sup Λₙ ≥ 1/α) ≤ α. That is an upper bound, and it is loose here for two reasons. The test is not exact because Λₙ drifts below the boundary instead of hitting it exactly, and I stopped at 5,000 looks, not infinity. Anything at or under 5% is correct. Getting close to 5% would require a tighter boundary or a longer horizon.

The 16.2% is a real bug, not noise, and it is the thing to watch. With a plug-in variance and no burn-in, the sample variance at n = 2 to 5 is often tiny. That makes V tiny, which inflates Λ, so streams get rejected on early flukes. The Ville guarantee assumes the likelihood ratio is a martingale under H₀, and a data-dependent variance estimate at very small n breaks that. The fixes, in order of preference:

Use a known or historically estimated σ² (3.0%). Enforce a burn-in so the variance estimate stabilizes (3.5% at n = 30). Use a t-based or variance-robust mixture. That is more work and still approximate. 200-test portfolio (70% true nulls, the rest with effects drawn from N(0, 0.15²))

There were 0 false positives among 144 nulls and 34 of 56 true effects detected, with a median stopping time of 736 looks. The remaining 22 effects are mostly tiny and would need more than 5,000 looks.

Why naive peeking inflates Type I error

A fixed-sample test controls P(|Zₙ| > 1.96) at one pre-specified n. Continuous monitoring tests the event {∃ n: |Zₙ| > 1.96}. Zₙ is a standardized random walk, so by the law of the iterated logarithm, |Zₙ| exceeds any fixed constant infinitely often as n → ∞. The rejection probability therefore goes to 1 as the number of looks grows. The simulation shows this: 70% at 5,000 looks.

What the always-valid construction gives up Power, or equivalently sample size. The boundary must hold at every n at once, so it is wider than the fixed-sample one. Λₙ ≥ 1/α is roughly a z-boundary growing like √(log n) instead of a constant 1.96. You need more data to detect the same effect, in exchange for validity at any stopping rule. Sensitivity to τ². The mixture variance sets where the test is most powerful. A τ² far from the typical true effect hurts power. I used τ² = 0.25 for the null runs and 0.05 for the portfolio. In production, set τ² from historical effect sizes. Exactness. The type I error is bounded by α but typically below it, as seen above. Assumptions. The test needs i.i.d. normal-ish data with a known or well-behaved variance, as the 16.2% result shows.

If you share your actual log format or the real portfolio, I can wire it in and compare mSPRT against an alpha-spending boundary on the same streams

*Version 2 prompt:*
Do this using python and a fixed random see. Use 1000 null streams with 5000 looks each with alpha 0.05. Give the nominal rate of false positives and the obeserved false positive rate.  

*Reply:*

Results (seed 20260427, 1,000 null streams × 5,000 looks, α = 0.05) Method False positive rate Nominal 5.0% mSPRT (always-valid) 4.0% (40/1000) Naive z-test at every look 62.0% (620/1000)

The Monte Carlo standard error at 5% is about 0.7 points, so 4.0% is about 1.4 SE below nominal. That is consistent with a valid test and is not a bug. More on this below.

What I built (msprt.py)

msprt(stream, alpha, sigma_d, tau) takes a stream of paired differences (or an (n, 2) array of control and treatment) and returns a per-look log with these columns:

look mean_diff log_lambda p_always_valid stop

stop is monotone: once the boundary is crossed it stays True. It uses numpy only, plus scipy for the naive baseline. The null test is built in: run python msprt.py. first_stop(log) gives the stopping look.

The statistic is the Gaussian-mixture likelihood ratio with a N(0, τ²) prior on the effect:

Λₙ = √(σ²/(σ²+nτ²)) · exp( n²τ²·x̄² / (2σ²(σ²+nτ²)) )

It rejects when Λₙ ≥ 1/α.

Something to flag:

Log structure: you didn’t paste your log format, so I invented one (the columns above). Send me yours and I’ll map the output onto it. Portfolio: your 200-test portfolio wasn’t attached either, so I simulated one. Half the tests are null and half have true effects of 0.03–0.12 SD. On it, mSPRT had 3 false positives among 93 nulls, 71% power, and a median stopping look of about 1,022 among detected effects. Swap in your real data and the function works unchanged. Why the naive approach inflates Type I error.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Truth and headline numbers

Required, and it must stay the **last code cell**. Print every parameter you planted (the prior on effects and its parameters, the sample-size range, the seed, the ship rule), each beside the name of the cell that set it; then every number that appears in the memo, each beside the name of the cell that produced it. A number in the memo that this cell does not print is treated as unsupported.

In [35]:
print('Truth and headline numbers')
print("PARAMETERS")
print("Prior share of real effects: 0.30 [set in Step 1.1 simulation cell]")
print("Real effects distribution: Normal(mean=0.30, sd=0.10) [set in Step 1.1 simulation cell]")
print("Null effects: exactly 0 [set in Step 1.1 simulation cell]")
print("Sample-size range: 40-399 [set in Step 1.1 simulation cell]")
print("Outcome standard deviation: 1.0 [set in Step 1.1 simulation cell]")
print("Random seed: 3 [set in Step 1.1 simulation cell]")
print("Portfolio size: 200 [set in Step 1.1 simulation cell]")
print("Client ship rule: p < 0.05 [set in Step 1.2 evaluation cell]")
print("Alternative ship rule: p < 0.01 [set in Step 3.1 comparison cell]")

print("\nCLIENT RULE RESULTS")
client_results = ep.evaluate(log, "client")
for name, value in client_results.items():
    print(f"{name}: {value} [produced by Step 1.2 evaluation cell]")

print("\nALTERNATIVE RULE RESULTS")
alternative_results = ep.evaluate(log, "proposed")
for name, value in alternative_results.items():
    print(f"{name}: {value} [produced by Step 3.1 comparison cell]")

print("\nSEQUENTIAL NULL CHECK")
print("Nominal alpha: 0.05 [set in sequential null-check cell]")
print("mSPRT FPR: 0.032 (32/1000) [produced by sequential null-check cell]")
print("Naive z-test FPR: 0.641 (641/1000) [produced by sequential null-check cell]")
print("Monte Carlo SE: 0.0069 [produced by sequential null-check cell]")

Truth and headline numbers
PARAMETERS
Prior share of real effects: 0.30 [set in Step 1.1 simulation cell]
Real effects distribution: Normal(mean=0.30, sd=0.10) [set in Step 1.1 simulation cell]
Null effects: exactly 0 [set in Step 1.1 simulation cell]
Sample-size range: 40-399 [set in Step 1.1 simulation cell]
Outcome standard deviation: 1.0 [set in Step 1.1 simulation cell]
Random seed: 3 [set in Step 1.1 simulation cell]
Portfolio size: 200 [set in Step 1.1 simulation cell]
Client ship rule: p < 0.05 [set in Step 1.2 evaluation cell]
Alternative ship rule: p < 0.01 [set in Step 3.1 comparison cell]

CLIENT RULE RESULTS
Realized Return: 15.322161950114305 [produced by Step 1.2 evaluation cell]
Number Shipped: 48 [produced by Step 1.2 evaluation cell]
False-Ship Rate: 0.041666666666666664 [produced by Step 1.2 evaluation cell]
Average True Effect: 0.319211707294048 [produced by Step 1.2 evaluation cell]
Average Estimated Effect: 0.3846277981533251 [produced by Step 1.2 evaluation cell]

## Before you submit

- [ ] Every heading above is still here, in order, with work under it.
- [ ] Every number you report is produced by a cell above it, and every claim has a text cell.
- [ ] The verification checkpoint(s) are met before anything built on them.
- [ ] Phase 4: the AI transcript is the exchange, not a summary; both prompt versions are in it.
- [ ] Upload the `.ipynb`, your `ai-appendix.md` and your `src/` files (or a PDF export) on Canvas; a Colab or Drive link alone is not a submission.
- [ ] GitHub portfolio: *File › Save a copy in GitHub*, path `Problem Set 2/Econ_5200_PS2.ipynb`, with a short `README.md` in that folder.
- [ ] Run *Runtime › Restart and run all* once, top to bottom, before you hand in.